In [1]:
import pandas as pd
import yaml
from sqlalchemy import create_engine

In [2]:
import os
config_path = 'config.yml' if os.path.exists('config.yml') else os.path.join('..', 'config.yml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config_etl = config['ETL_PRO']

url = (
    f"{config_etl['drivername']}://"
    f"{config_etl['user']}:{config_etl['password']}@"
    f"{config_etl['host']}:{config_etl['port']}/"
    f"{config_etl['dbname']}"
)

etl_conn = create_engine(url)

In [3]:
hecho_servicios = pd.read_sql_table('hecho_servicios', etl_conn)
hecho_novedades = pd.read_sql_table('hecho_novedades', etl_conn)

dim_fecha = pd.read_sql_table('dim_fecha', etl_conn)
dim_hora = pd.read_sql_table('dim_hora', etl_conn)
dim_proveedor = pd.read_sql_table('dim_proveedor', etl_conn)
dim_mensajero = pd.read_sql_table('dim_mensajero', etl_conn)
dim_sede = pd.read_sql_table('dim_sede', etl_conn)
dim_novedades = pd.read_sql_table('dim_novedades', etl_conn)

## Pregunta 1

In [4]:
consulta = """
SELECT
    df.nombre_mes,
    COUNT(*) AS total_servicios
FROM hecho_servicios hs
JOIN dim_fecha df
    ON hs.key_dim_fecha_solicitud = df.key_fecha
GROUP BY df.nombre_mes
ORDER BY total_servicios DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,nombre_mes,total_servicios
0,Mayo,4711
1,Julio,4542
2,Abril,4479
3,Agosto,4297
4,Junio,4180
5,Marzo,3336
6,Febrero,2477
7,Enero,291
8,Diciembre,25
9,Noviembre,17


## Pregunta 2

In [5]:
consulta = """
SELECT
    df.nombre_dia,
    COUNT(*) AS total_servicios
FROM hecho_servicios hs
JOIN dim_fecha df
    ON hs.key_dim_fecha_solicitud = df.key_fecha
GROUP BY df.nombre_dia
ORDER BY total_servicios DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,nombre_dia,total_servicios
0,Martes,5393
1,Viernes,5271
2,Jueves,5146
3,Miércoles,4958
4,Lunes,4302
5,Sábado,2473
6,Domingo,840


## Pregunta 3

In [6]:
consulta = """
SELECT 
    CASE 
        WHEN hora_num BETWEEN 0 AND 2 THEN '00-02'
        WHEN hora_num BETWEEN 3 AND 5 THEN '03-05'
        WHEN hora_num BETWEEN 6 AND 8 THEN '06-08'
        WHEN hora_num BETWEEN 9 AND 11 THEN '09-11'
        WHEN hora_num BETWEEN 12 AND 14 THEN '12-14'
        WHEN hora_num BETWEEN 15 AND 17 THEN '15-17'
        WHEN hora_num BETWEEN 18 AND 20 THEN '18-20'
        ELSE '21-23'
    END AS rango_hora,
    COUNT(*) AS total_servicios_activos
FROM (
    SELECT 
        EXTRACT(HOUR FROM gs) AS hora_num
    FROM hecho_servicios hs
    LEFT JOIN dim_fecha fa ON hs.key_dim_fecha_mensajero_asignado = fa.key_fecha
    LEFT JOIN dim_hora ha ON hs.key_dim_hora_mensajero_asignado = ha.key_hora
    LEFT JOIN dim_fecha fc ON hs.key_dim_fecha_finalizado_completo = fc.key_fecha
    LEFT JOIN dim_hora hc ON hs.key_dim_hora_finalizado_completo = hc.key_hora
    CROSS JOIN LATERAL generate_series(
        date_trunc('hour', fa.fecha_completa + ha.hora_completa),
        date_trunc('hour', fc.fecha_completa + hc.hora_completa),
        '1 hour'::interval
    ) gs
    WHERE fa.fecha_completa IS NOT NULL AND fc.fecha_completa IS NOT NULL
) horas_ocupadas
GROUP BY 1 
ORDER BY total_servicios_activos DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,rango_hora,total_servicios_activos
0,09-11,14245
1,12-14,12577
2,15-17,12174
3,06-08,9638
4,18-20,8266
5,21-23,8077
6,00-02,7991
7,03-05,7950


## Pregunta 4

In [7]:
consulta = """
SELECT
    dp.nombre_proveedor,
    df.nombre_mes,
    COUNT(*) AS total_servicios
FROM hecho_servicios hs
JOIN dim_proveedor dp
    ON hs.key_dim_proveedor = dp.key_proveedor
JOIN dim_fecha df
    ON hs.key_dim_fecha_solicitud = df.key_fecha
GROUP BY
    dp.nombre_proveedor,
    df.nombre_mes
ORDER BY
    dp.nombre_proveedor,
    df.nombre_mes;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,nombre_proveedor,nombre_mes,total_servicios
0,BANCO REGIONAL DE SANGRE BLOD-LIFE,Abril,50
1,BANCO REGIONAL DE SANGRE BLOD-LIFE,Agosto,31
2,BANCO REGIONAL DE SANGRE BLOD-LIFE,Febrero,17
3,BANCO REGIONAL DE SANGRE BLOD-LIFE,Julio,23
4,BANCO REGIONAL DE SANGRE BLOD-LIFE,Junio,24
...,...,...,...
102,UNIDAD DE TRAUMA DEL OESTE,Febrero,76
103,UNIDAD DE TRAUMA DEL OESTE,Julio,97
104,UNIDAD DE TRAUMA DEL OESTE,Junio,102
105,UNIDAD DE TRAUMA DEL OESTE,Marzo,85


## Pregunta 5

In [8]:
consulta = """
SELECT
    dm.nombre,
    COUNT(*) AS servicios_realizados
FROM hecho_servicios hs
JOIN dim_mensajero dm
    ON hs.key_mensajero_1 = dm.key_mensajero
GROUP BY dm.nombre
ORDER BY servicios_realizados DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,nombre,servicios_realizados
0,Jhon Tróchez,2439
1,Sebastián Acuña,1548
2,Juan Solano,1510
3,Andrés Gutiérrez,1456
4,Héctor Aquiles,1348
5,Jan Sastre,1333
6,Luis Cardona,1327
7,Jairon Montes,1252
8,J. Pedroza,1249
9,Jhon Muñoz,1228


## Pregunta 6

In [9]:
consulta = """
SELECT
    dp.nombre_proveedor,
    ds.nombre,
    COUNT(*) AS total_servicios
FROM hecho_servicios hs
JOIN dim_proveedor dp
    ON hs.key_dim_proveedor = dp.key_proveedor
JOIN dim_sede ds
    ON hs.key_sede = ds.key_sede
GROUP BY
    dp.nombre_proveedor,
    ds.nombre
ORDER BY
    dp.nombre_proveedor,
    total_servicios DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,nombre_proveedor,nombre,total_servicios
0,CARROS DEL PACIFICO (CHINA),PRINCIPAL NORTE / CHINA PACIFICO,972
1,CARROS DEL PACIFICO (CHINA),BUSES Y CAMIONES -,960
2,CARROS DEL PACIFICO (CHINA),MEGATALLER / CHINA PACIFICO,853
3,CARROS DEL PACIFICO (CHINA),PALMIRA BODEGA 20 /,331
4,CARROS DEL PACIFICO (CHINA),TULUA BODEGA 26 /,208
5,CARROS DEL PACIFICO (CHINA),POPAYAN BODEGA 28 / A,124
6,CARROS DEL PACIFICO (CHINA),AUTOPISTA,34
7,CARROS DEL PACIFICO (CHINA),JMC /,33
8,CARROS DEL PACIFICO (CHINA),PASTO BODEGA 29/,33
9,CARROS DEL PACIFICO (CHINA),FORDES PASOANCHOS,14


## Pregunta 7

In [10]:
consulta = """
SELECT
    AVG(
        (ff.fecha_completa + hf.hora_completa)
        -
        (fs.fecha_completa + hs_time.hora_completa)
    ) AS tiempo_promedio
FROM hecho_servicios hs
JOIN dim_fecha fs ON hs.key_dim_fecha_solicitud = fs.key_fecha
JOIN dim_hora hs_time ON hs.key_dim_hora_solicitud = hs_time.key_hora
JOIN dim_fecha ff ON hs.key_dim_fecha_finalizado_completo = ff.key_fecha
JOIN dim_hora hf ON hs.key_dim_hora_finalizado_completo = hf.key_hora;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,tiempo_promedio
0,0 days 12:09:40.705271


## Pregunta 8

In [11]:
consulta = """
WITH promedios AS (
    SELECT
        ROUND(CAST(EXTRACT(EPOCH FROM AVG((fa.fecha_completa + ha.hora_completa) - (fi.fecha_completa + hi.hora_completa))) / 60 AS NUMERIC), 2) AS iniciado_asignado,
        ROUND(CAST(EXTRACT(EPOCH FROM AVG((fr.fecha_completa + hr.hora_completa) - (fa.fecha_completa + ha.hora_completa))) / 60 AS NUMERIC), 2) AS asignado_recogido,
        ROUND(CAST(EXTRACT(EPOCH FROM AVG((fe.fecha_completa + he.hora_completa) - (fr.fecha_completa + hr.hora_completa))) / 60 AS NUMERIC), 2) AS recogido_entregado,
        ROUND(CAST(EXTRACT(EPOCH FROM AVG((fc.fecha_completa + hc.hora_completa) - (fe.fecha_completa + he.hora_completa))) / 60 AS NUMERIC), 2) AS entregado_cerrado
    FROM hecho_servicios hs
    LEFT JOIN dim_fecha fi ON hs.key_dim_fecha_iniciado = fi.key_fecha
    LEFT JOIN dim_hora hi ON hs.key_dim_hora_iniciado = hi.key_hora
    LEFT JOIN dim_fecha fa ON hs.key_dim_fecha_mensajero_asignado = fa.key_fecha
    LEFT JOIN dim_hora ha ON hs.key_dim_hora_mensajero_asignado = ha.key_hora
    LEFT JOIN dim_fecha fr ON hs.key_dim_fecha_recogido_mensajero = fr.key_fecha
    LEFT JOIN dim_hora hr ON hs.key_dim_hora_recogido_mensajero = hr.key_hora
    LEFT JOIN dim_fecha fe ON hs.key_dim_fecha_entregado = fe.key_fecha
    LEFT JOIN dim_hora he ON hs.key_dim_hora_entregado = he.key_hora
    LEFT JOIN dim_fecha fc ON hs.key_dim_fecha_finalizado_completo = fc.key_fecha
    LEFT JOIN dim_hora hc ON hs.key_dim_hora_finalizado_completo = hc.key_hora
)
SELECT 
    iniciado_asignado,
    asignado_recogido,
    recogido_entregado,
    entregado_cerrado,
    CASE GREATEST(iniciado_asignado, asignado_recogido, recogido_entregado, entregado_cerrado)
        WHEN iniciado_asignado THEN 'iniciado_asignado'
        WHEN asignado_recogido THEN 'asignado_recogido'
        WHEN recogido_entregado THEN 'recogido_entregado'
        WHEN entregado_cerrado THEN 'entregado_cerrado'
    END AS etapa_mas_demorada
FROM promedios;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,iniciado_asignado,asignado_recogido,recogido_entregado,entregado_cerrado,etapa_mas_demorada
0,149.79,95.37,99.91,293.49,entregado_cerrado


## Pregunta 9

In [12]:
consulta = """
SELECT
    grupo_novedad,
    COUNT(*) AS total_novedades
FROM dim_novedades
GROUP BY grupo_novedad
ORDER BY total_novedades DESC;
"""

resultado = pd.read_sql(consulta, etl_conn)
resultado

,grupo_novedad,total_novedades
0,Otros / Comentarios Varios,1649
1,Espera de Turno / Fila / Trámite,1209
2,Espera de Alistamiento / Facturación,1023
3,Reasignación / Trasbordo de Mensajero,639
4,Cliente / Contacto No Disponible,210
5,Error en Datos / Tipo de Vehículo,179
6,Establecimiento Cerrado / Almuerzo,148
7,Cancelado / Reprogramado,93
8,Problema de Vehículo / Tránsito,58


In [ ]:
sql = """
SELECT
    dn.grupo_novedad,
    SUM(hn.cantidad_novedades) AS total_novedades
FROM hecho_novedades hn
JOIN dim_novedades dn
    ON hn.key_novedad = dn.key_novedad
GROUP BY dn.grupo_novedad
ORDER BY total_novedades DESC;
"""
resultado = pd.read_sql(sql, etl_conn)
resultado